# LangChain 的 Redis 缓存

本 Notebook 演示了如何使用 `langchain-redis` 包中的 `RedisCache` 和 `RedisSemanticCache` 类来实现 LLM 响应的缓存。

## 设置

首先，让我们安装所需的依赖项，并确保我们有一个正在运行的 Redis 实例。

In [ ]:
%pip install -U langchain-core langchain-redis langchain-openai redis

确保您运行着一个 Redis 服务器。您可以使用 Docker 启动一个：

```
docker run -d -p 6379:6379 redis:latest
```

或者根据您的操作系统说明，在本地安装并运行 Redis。

In [2]:
import os

# Use the environment variable if set, otherwise default to localhost
REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379")
print(f"Connecting to Redis at: {REDIS_URL}")

Connecting to Redis at: redis://redis:6379


## 导入所需库

In [3]:
import time

from langchain.globals import set_llm_cache
from langchain.schema import Generation
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_redis import RedisCache, RedisSemanticCache

In [4]:
import langchain_core
import langchain_openai
import openai
import redis

### 设置 OpenAI API 密钥

You need an OpenAI API key to use the OpenAI integration. You can get one from [the OpenAI website](https://platform.openai.com/account/api-keys).

您需要一个 OpenAI API 密钥才能使用 OpenAI 集成。您可以从 [OpenAI 网站](https://platform.openai.com/account/api-keys) 获取。

Once you have your API key, you can set it as an environment variable.

获取 API 密钥后，您可以将其设置为环境变量。

**Linux/macOS**

```bash
export OPENAI_API_KEY="YOUR_API_KEY"
```

**Windows (Command Prompt)**

```cmd
set OPENAI_API_KEY="YOUR_API_KEY"
```

**Windows (PowerShell)**

```powershell
$env:OPENAI_API_KEY="YOUR_API_KEY"
```

Replace `"YOUR_API_KEY"` with your actual API key.

将 `"YOUR_API_KEY"` 替换为您的实际 API 密钥。

Once set, you can use the OpenAI integration in your application.

设置完成后，您就可以在应用程序中使用 OpenAI 集成了。

In [5]:
from getpass import getpass

# Check if OPENAI_API_KEY is already set in the environment
openai_api_key = os.getenv("OPENAI_API_KEY")

if not openai_api_key:
    print("OpenAI API key not found in environment variables.")
    openai_api_key = getpass("Please enter your OpenAI API key: ")

    # Set the API key for the current session
    os.environ["OPENAI_API_KEY"] = openai_api_key
    print("OpenAI API key has been set for this session.")
else:
    print("OpenAI API key found in environment variables.")

OpenAI API key not found in environment variables.


Please enter your OpenAI API key:  ········


OpenAI API key has been set for this session.


## 使用 RedisCache

RedisCache 是一个用于缓存数据的分布式内存数据库。本指南将介绍如何使用 RedisCache。

### 安装

您可以使用 npm 或 yarn 来安装 RedisCache：

```bash
npm install redis-cache
# 或者
yarn add redis-cache
```

### 基本用法

以下是如何使用 RedisCache 的基本示例：

```javascript
import RedisCache from 'redis-cache';

// 创建一个 RedisCache 实例
const cache = new RedisCache({
  port: 6379, // Redis 服务器端口
  host: 'localhost', // Redis 服务器主机
  // password: 'your_password', // 如果您的 Redis 服务器需要密码
});

// 设置一个键值对
await cache.set('mykey', 'myvalue');

// 获取一个键的值
const value = await cache.get('mykey');
console.log(value); // 输出: myvalue

// 删除一个键
await cache.delete('mykey');

// 检查一个键是否存在
const exists = await cache.has('mykey');
console.log(exists); // 输出: false
```

### 高级用法

#### TTL (Time To Live)

您可以为缓存项设置过期时间（TTL）。

```javascript
// 设置一个键值对，并设置 60 秒的过期时间
await cache.set('expiring_key', 'some_data', { ttl: 60 });
```

#### 命名空间

使用命名空间可以帮助您组织缓存数据，避免键冲突。

```javascript
// 创建一个带有命名空间的缓存实例
const namespacedCache = cache.namespace('user_data');

await namespacedCache.set('user:123', { name: 'Alice' });
const userData = await namespacedCache.get('user:123');
console.log(userData);
```

#### 事件

RedisCache 触发各种事件，您可以监听这些事件以响应缓存操作。

```javascript
cache.on('set', (key, value) => {
  console.log(`Key "${key}" set with value "${value}"`);
});

cache.on('get', (key, value) => {
  console.log(`Key "${key}" retrieved value "${value}"`);
});

cache.on('delete', (key) => {
  console.log(`Key "${key}" deleted`);
});

cache.on('error', (err) => {
  console.error('RedisCache error:', err);
});
```

### 配置选项

创建 `RedisCache` 实例时，您可以传递一个配置对象来定制其行为：

*   `port` (number): Redis 服务器的端口。默认为 `6379`。
*   `host` (string): Redis 服务器的主机名或 IP 地址。默认为 `'localhost'`。
*   `password` (string, optional): 如果您的 Redis 服务器需要密码进行身份验证，请在此处提供。
*   `db` (number): 要使用的 Redis 数据库编号。默认为 `0`。
*   `prefix` (string, optional): 添加到所有缓存键的前缀。
*   `namespaceSeparator` (string): 用于分隔命名空间和键的字符。默认为 `':'`。

### 错误处理

在与 Redis 交互时可能会发生错误。建议您使用 `try...catch` 块来处理潜在的错误，或者监听 `'error'` 事件。

```javascript
try {
  await cache.set('test', 'data');
} catch (error) {
  console.error('Failed to set cache:', error);
}
```

### 结论

RedisCache 是一个强大且易于使用的 Redis 客户端库，适用于需要缓存数据的 Node.js 应用程序。通过其简单的 API 和高级功能，您可以轻松地将 Redis 集成到您的项目中。

In [6]:
# Initialize RedisCache
redis_cache = RedisCache(redis_url=REDIS_URL)

# Set the cache for LangChain to use
set_llm_cache(redis_cache)

# Initialize the language model
llm = OpenAI(temperature=0)


# Function to measure execution time
def timed_completion(prompt):
    start_time = time.time()
    result = llm.invoke(prompt)
    end_time = time.time()
    return result, end_time - start_time


# First call (not cached)
prompt = "Explain the concept of caching in three sentences."
result1, time1 = timed_completion(prompt)
print(f"First call (not cached):\nResult: {result1}\nTime: {time1:.2f} seconds\n")

# Second call (should be cached)
result2, time2 = timed_completion(prompt)
print(f"Second call (cached):\nResult: {result2}\nTime: {time2:.2f} seconds\n")

print(f"Speed improvement: {time1 / time2:.2f}x faster")

# Clear the cache
redis_cache.clear()
print("Cache cleared")

First call (not cached):
Result: 

Caching is the process of storing frequently accessed data in a temporary storage location for faster retrieval. This helps to reduce the time and resources needed to access the data from its original source. Caching is commonly used in computer systems, web browsers, and databases to improve performance and efficiency.
Time: 1.16 seconds

Second call (cached):
Result: 

Caching is the process of storing frequently accessed data in a temporary storage location for faster retrieval. This helps to reduce the time and resources needed to access the data from its original source. Caching is commonly used in computer systems, web browsers, and databases to improve performance and efficiency.
Time: 0.05 seconds

Speed improvement: 25.40x faster
Cache cleared


## 使用 RedisSemanticCache

This guide explains how to use RedisSemanticCache, a semantic cache for Redis.

### What is RedisSemanticCache?

RedisSemanticCache is a semantic cache that stores embeddings and their associated metadata in Redis. It allows you to quickly retrieve similar embeddings based on a given query embedding. This is useful for a variety of applications, such as:

* **Recommendation systems:** Recommend similar items to users based on their past behavior.
* **Search engines:** Improve search relevance by finding documents that are semantically similar to the query.
* **Question answering systems:** Find relevant answers to user questions by matching them with similar questions in a knowledge base.

### How to use RedisSemanticCache

To use RedisSemanticCache, you need to have Redis installed and running. You also need to have the `redis-py` library installed.

```bash
pip install redis-py
```

Here's a basic example of how to use RedisSemanticCache:

```python
import redis
from redis_semantic_cache import RedisSemanticCache

# Connect to Redis
redis_client = redis.Redis(host='localhost', port=6379, db=0)

# Initialize RedisSemanticCache
cache = RedisSemanticCache(redis_client)

# Add an embedding to the cache
embedding = [0.1, 0.2, 0.3]
metadata = {"text": "This is a sample text."}
cache.add_embedding(embedding, metadata)

# Retrieve similar embeddings
query_embedding = [0.15, 0.25, 0.35]
similar_embeddings = cache.get_similar_embeddings(query_embedding)

# Print the results
for embedding, metadata in similar_embeddings:
    print(f"Embedding: {embedding}, Metadata: {metadata}")
```

### Advanced Usage

RedisSemanticCache also supports advanced features such as:

* **Filtering:** Filter embeddings based on metadata.
* **Pagination:** Retrieve embeddings in pages.
* **Tuning:** Adjust the similarity threshold and other parameters.

For more information, please refer to the [RedisSemanticCache documentation](https://github.com/redis/redis-semantic-cache).

In [7]:
# Initialize RedisSemanticCache
embeddings = OpenAIEmbeddings()
semantic_cache = RedisSemanticCache(
    redis_url=REDIS_URL, embeddings=embeddings, distance_threshold=0.2
)

# Set the cache for LangChain to use
set_llm_cache(semantic_cache)


# Function to test semantic cache
def test_semantic_cache(prompt):
    start_time = time.time()
    result = llm.invoke(prompt)
    end_time = time.time()
    return result, end_time - start_time


# Original query
original_prompt = "What is the capital of France?"
result1, time1 = test_semantic_cache(original_prompt)
print(
    f"Original query:\nPrompt: {original_prompt}\nResult: {result1}\nTime: {time1:.2f} seconds\n"
)

# Semantically similar query
similar_prompt = "Can you tell me the capital city of France?"
result2, time2 = test_semantic_cache(similar_prompt)
print(
    f"Similar query:\nPrompt: {similar_prompt}\nResult: {result2}\nTime: {time2:.2f} seconds\n"
)

print(f"Speed improvement: {time1 / time2:.2f}x faster")

# Clear the semantic cache
semantic_cache.clear()
print("Semantic cache cleared")

Original query:
Prompt: What is the capital of France?
Result: 

The capital of France is Paris.
Time: 1.52 seconds

Similar query:
Prompt: Can you tell me the capital city of France?
Result: 

The capital of France is Paris.
Time: 0.29 seconds

Speed improvement: 5.22x faster
Semantic cache cleared


## 高级用法

This section covers advanced usage patterns for the `useQuery` hook.

本节将介绍 `useQuery` hook 的高级用法模式。

### Fetching Data with Variables

`useQuery` can accept variables to customize your GraphQL queries.

### 使用变量获取数据

`useQuery` 可以接受变量来定制你的 GraphQL 查询。

```js
import { useQuery, gql } from '@apollo/client';

const GET_GREETING = gql`
  query GetGreeting($language: String!) {
    greeting(language: $language) {
      message
    }
  }
`;

function Hello({ language }) {
  const { loading, error, data } = useQuery(GET_GREETING, {
    variables: { language },
  });

  if (loading) return <p>Loading...</p>;
  if (error) return <p>Error :(</p>;

  return <h1>{data.greeting.message}</h1>;
}
```

### Polling

You can configure `useQuery` to poll your GraphQL server at a specified interval.

### 轮询

你可以配置 `useQuery` 以指定的间隔轮询你的 GraphQL 服务器。

```js
import { useQuery, gql } from '@apollo/client';

const GET_MESSAGES = gql`
  query GetMessages {
    messages {
      id
      content
    }
  }
`;

function Messages() {
  const { loading, error, data } = useQuery(GET_MESSAGES, {
    pollInterval: 500, // Refetch every 500ms
  });

  if (loading) return <p>Loading...</p>;
  if (error) return <p>Error :(</p>;

  return data.messages.map(({ id, content }) => (
    <p key={id}>{content}</p>
  ));
}
```

### Fetching More Data

When you need to fetch more data for a query that has already been executed, you can use the `fetchMore` function returned by `useQuery`.

### 获取更多数据

当你需要为已执行的查询获取更多数据时，可以使用 `useQuery` 返回的 `fetchMore` 函数。

```js
import { useQuery, gql } from '@apollo/client';

const GET_BOOKS = gql`
  query GetBooks($cursor: String) {
    books(first: 3, after: $cursor) {
      edges {
        node {
          id
          title
        }
      }
      pageInfo {
        endCursor
        hasNextPage
      }
    }
  }
`;

function BookList() {
  const { loading, error, data, fetchMore } = useQuery(GET_BOOKS, {
    variables: { cursor: null },
    // Only display loading message once
    notifyOnNetworkStatusChange: true,
  });

  if (error) return <p>Error :(</p>;

  const books = data?.books?.edges?.map(edge => edge.node) || [];
  const { endCursor, hasNextPage } = data?.books?.pageInfo || {};

  return (
    <div>
      <ul>
        {books.map(book => (
          <li key={book.id}>{book.title}</li>
        ))}
      </ul>
      <button
        onClick={() =>
          fetchMore({
            variables: {
              // Move the cursor to the end of the current list
              cursor: endCursor,
            },
            // // Transform the incoming data
            // updateQuery: (prevResult, { fetchMoreResult }) => {
            //   if (!fetchMoreResult) return prevResult;
            //   return {
            //     books: {
            //       ...fetchMoreResult.books,
            //       edges: [
            //         ...(prevResult.books?.edges || []),
            //         ...(fetchMoreResult.books?.edges || []),
            //       ],
            //     },
            //   };
            // },
          })
        }
        disabled={!hasNextPage || loading}
      >
        {loading ? 'Loading more...' : 'Load More'}
      </button>
    </div>
  );
}
```

### Refetching Data

You can manually refetch data for a query using the `refetch` function returned by `useQuery`.

### 重新获取数据

你可以使用 `useQuery` 返回的 `refetch` 函数手动重新获取查询数据。

```js
import { useQuery, gql } from '@apollo/client';

const GET_LAUNCHES = gql`
  query GetLaunches {
    launchesPast(limit: 10) {
      id
      mission_name
      launch_date_local
    }
  }
`;

function Launches() {
  const { loading, error, data, refetch } = useQuery(GET_LAUNCHES);

  if (loading) return <p>Loading...</p>;
  if (error) return <p>Error :(</p>;

  return (
    <div>
      {data.launchesPast.map(({ id, mission_name, launch_date_local }) => (
        <p key={id}>
          {mission_name} was launched on {launch_date_local}
        </p>
      ))}
      <button onClick={() => refetch()}>Refetch launches</button>
    </div>
  );
}
```

### Optimistic Updates

Optimistic updates allow you to update the UI immediately after a mutation, before the server has responded. This can make your application feel more responsive.

### 乐观更新

乐观更新允许你在执行突变后立即更新 UI，而无需等待服务器响应。这可以使你的应用程序感觉更具响应性。

```js
import { useQuery, useMutation, gql } from '@apollo/client';

const GET_TODOS = gql`
  query GetTodos {
    todos {
      id
      text
      completed
    }
  }
`;

const ADD_TODO = gql`
  mutation AddTodo($text: String!) {
    addTodo(text: $text) {
      id
      text
      completed
    }
  }
`;

function TodoApp() {
  const { loading, error, data } = useQuery(GET_TODOS);
  const [addTodo] = useMutation(ADD_TODO, {
    // Optimistically update the cache
    optimisticResponse: {
      __typename: 'Mutation',
      addTodo: {
        __typename: 'Todo',
        id: 'temp-id', // Temporary ID
        text: 'New todo',
        completed: false,
      },
    },
    update(cache, { data: { addTodo } }) {
      const existingTodos = cache.readQuery({ query: GET_TODOS });
      cache.writeQuery({
        query: GET_TODOS,
        data: { todos: [...existingTodos.todos, addTodo] },
      });
    },
  });

  if (loading) return <p>Loading...</p>;
  if (error) return <p>Error :(</p>;

  return (
    <div>
      <ul>
        {data.todos.map(({ id, text, completed }) => (
          <li key={id} style={{ textDecoration: completed ? 'line-through' : 'none' }}>
            {text}
          </li>
        ))}
      </ul>
      <button onClick={() => addTodo({ variables: { text: 'New todo' } })}>
        Add Todo
      </button>
    </div>
  );
}
```

### Error Handling

`useQuery` returns an `error` object that you can use to display error messages to your users.

### 错误处理

`useQuery` 返回一个 `error` 对象，你可以使用它向用户显示错误消息。

```js
import { useQuery, gql } from '@apollo/client';

const GET_DATA = gql`
  query GetData {
    data {
      message
    }
  }
`;

function MyComponent() {
  const { loading, error, data } = useQuery(GET_DATA);

  if (loading) return <p>Loading...</p>;
  if (error) return <p>Oh no, something went wrong: {error.message}</p>;

  return <p>{data.data.message}</p>;
}
```

### Conditional Fetching

You can conditionally fetch data by passing `skip: true` to the `useQuery` options.

### 条件获取

你可以通过将 `skip: true` 传递给 `useQuery` 选项来有条件地获取数据。

```js
import { useQuery, gql } from '@apollo/client';

const GET_USER_DATA = gql`
  query GetUserData($userId: ID!) {
    user(id: $userId) {
      name
      email
    }
  }
`;

function UserProfile({ userId }) {
  const { loading, error, data } = useQuery(GET_USER_DATA, {
    variables: { userId },
    skip: !userId, // Only fetch if userId is provided
  });

  if (!userId) return <p>Please provide a user ID.</p>;
  if (loading) return <p>Loading...</p>;
  if (error) return <p>Error :(</p>;

  return (
    <div>
      <h1>{data.user.name}</h1>
      <p>{data.user.email}</p>
    </div>
  );
}
```

### Fetch Policy

The `fetchPolicy` option controls when Apollo Client fetches data from your GraphQL server.

### 获取策略

`fetchPolicy` 选项控制 Apollo Client 何时从你的 GraphQL 服务器获取数据。

- `cache-first` (default): Use cached data if available, otherwise fetch from server.
- `cache-and-network`: Return cached data immediately (if available), and then fetch from network.
- `network-first`: Fetch from network, use cached data if network fails.
- `network-only`: Always fetch from network.

```js
import { useQuery, gql } from '@apollo/client';

const GET_SETTINGS = gql`
  query GetSettings {
    settings {
      theme
    }
  }
`;

function Settings() {
  const { loading, error, data } = useQuery(GET_SETTINGS, {
    fetchPolicy: 'network-only', // Always fetch from network
  });

  if (loading) return <p>Loading...</p>;
  if (error) return <p>Error :(</p>;

  return <p>Theme: {data.settings.theme}</p>;
}
```

### Context

You can pass custom context to your queries using the `context` option. This context will be available in your GraphQL server's request handlers.

### 上下文

你可以使用 `context` 选项将自定义上下文传递给你的查询。此上下文将在你的 GraphQL 服务器的请求处理程序中可用。

```js
import { useQuery, gql } from '@apollo/client';

const GET_DATA_WITH_CONTEXT = gql`
  query GetDataWithContext {
    dataWithContext {
      message
    }
  }
`;

function MyComponentWithContext() {
  const { loading, error, data } = useQuery(GET_DATA_WITH_CONTEXT, {
    context: {
      customHeader: 'my-custom-value',
    },
  });

  if (loading) return <p>Loading...</p>;
  if (error) return <p>Error :(</p>;

  return <p>{data.dataWithContext.message}</p>;
}

### 自定义 TTL（生存时间）

In [8]:
# Initialize RedisCache with custom TTL
ttl_cache = RedisCache(redis_url=REDIS_URL, ttl=5)  # 60 seconds TTL

# Update a cache entry
ttl_cache.update("test_prompt", "test_llm", [Generation(text="Cached response")])

# Retrieve the cached entry
cached_result = ttl_cache.lookup("test_prompt", "test_llm")
print(f"Cached result: {cached_result[0].text if cached_result else 'Not found'}")

# Wait for TTL to expire
print("Waiting for TTL to expire...")
time.sleep(6)

# Try to retrieve the expired entry
expired_result = ttl_cache.lookup("test_prompt", "test_llm")
print(
    f"Result after TTL: {expired_result[0].text if expired_result else 'Not found (expired)'}"
)

Cached result: Cached response
Waiting for TTL to expire...
Result after TTL: Not found (expired)


### 自定义 RedisSemanticCache

RedisSemanticCache 是一个用于存储和检索语义缓存的类。以下是如何自定义它的方法：

#### 1. 使用自定义的 Redis 客户端

默认情况下，RedisSemanticCache 使用 `redis-py` 库的默认客户端。如果您想使用自定义的 Redis 客户端，例如连接到不同的 Redis 实例或使用特定的配置，可以这样做：

```python
from semantic_cache import RedisSemanticCache
from redis import Redis

# 创建一个自定义的 Redis 客户端
redis_client = Redis(host='localhost', port=6379, db=0, password='your_password')

# 使用自定义客户端初始化 RedisSemanticCache
cache = RedisSemanticCache(redis_client=redis_client)
```

#### 2. 更改缓存键的前缀

RedisSemanticCache 使用一个默认的前缀来存储缓存键。您可以更改此前缀以避免与其他 Redis 键发生冲突。

```python
from semantic_cache import RedisSemanticCache

# 使用自定义的前缀初始化 RedisSemanticCache
cache = RedisSemanticCache(prefix="my_app_cache:")
```

#### 3. 设置缓存过期时间

您可以为 RedisSemanticCache 中的每个缓存条目设置过期时间。这有助于管理 Redis 内存使用情况。

```python
from semantic_cache import RedisSemanticCache

# 设置缓存过期时间为 3600 秒 (1 小时)
cache = RedisSemanticCache(ttl=3600)
```

#### 4. 结合使用自定义选项

您可以结合使用上述所有选项来根据您的具体需求自定义 RedisSemanticCache。

```python
from semantic_cache import RedisSemanticCache
from redis import Redis

# 创建一个自定义的 Redis 客户端
redis_client = Redis(host='localhost', port=6379, db=0, password='your_password')

# 使用自定义客户端、自定义前缀和自定义过期时间初始化 RedisSemanticCache
cache = RedisSemanticCache(
    redis_client=redis_client,
    prefix="my_app_cache:",
    ttl=3600
)

In [9]:
# Initialize RedisSemanticCache with custom settings
custom_semantic_cache = RedisSemanticCache(
    redis_url=REDIS_URL,
    embeddings=embeddings,
    distance_threshold=0.1,  # Stricter similarity threshold
    ttl=3600,  # 1 hour TTL
    name="custom_cache",  # Custom cache name
)

# Test the custom semantic cache
set_llm_cache(custom_semantic_cache)

test_prompt = "What's the largest planet in our solar system?"
result, _ = test_semantic_cache(test_prompt)
print(f"Original result: {result}")

# Try a slightly different query
similar_test_prompt = "Which planet is the biggest in the solar system?"
similar_result, _ = test_semantic_cache(similar_test_prompt)
print(f"Similar query result: {similar_result}")

# Clean up
custom_semantic_cache.clear()

Original result: 

The largest planet in our solar system is Jupiter.
Similar query result: 

The largest planet in our solar system is Jupiter.


## 结论

本笔记本演示了如何使用 `langchain-redis` 包中的 `RedisCache` 和 `RedisSemanticCache`。这些缓存机制可以通过减少冗余的 API 调用并利用语义相似性进行智能缓存，从而显著提高基于 LLM 的应用程序的性能。基于 Redis 的实现为分布式系统中的缓存提供了一种快速、可扩展且灵活的解决方案。